In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# BlindDetection-V1 fixed N=4 real single-image smoke
Prepared but not executed. This is engineering image-only callback evidence with `science_denominator=0`; it consumes the already frozen N_dev=256 threshold through calibration terminal ZIP content semantics.

In [ ]:
from google.colab import userdata
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys
import torch
import uuid
import zipfile

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
PRODUCER_EXACT = '0ff9b054c7caeaf487c3488fbcd04164d4db2ad3'
RUN_ID = f"blind-detection-v1-callback-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex}"
CHECKOUT = Path('/content') / (RUN_ID + '-checkout')
LOCAL_ROOT = Path('/content') / (RUN_ID + '-local')
RUNTIME_ROOT = LOCAL_ROOT / 'runtime'
LOCAL_RESULT = LOCAL_ROOT / 'callback_result.json'
CURRENT_RGB_DIR = LOCAL_ROOT / 'current-rgb'
LOCAL_STDOUT = LOCAL_ROOT / 'runner.stdout.txt'
LOCAL_STDERR = LOCAL_ROOT / 'runner.stderr.txt'
CALIBRATION_RUNS = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-runs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/callback-runs')
TERMINAL_ZIP = DRIVE_OUTPUT_ROOT / f'{RUN_ID}.zip'
PUBLIC_N4_CONFIG = CHECKOUT / 'configs/blind_detection/blind_detection_v1_callback_n4.json'

if 'BLIND_CALLBACK_RUNNER_CALLS' not in globals():
    BLIND_CALLBACK_RUNNER_CALLS = 0
if 'BLIND_CALLBACK_TERMINAL_ZIP_WRITES' not in globals():
    BLIND_CALLBACK_TERMINAL_ZIP_WRITES = 0
if BLIND_CALLBACK_RUNNER_CALLS != 0 or BLIND_CALLBACK_TERMINAL_ZIP_WRITES != 0:
    raise RuntimeError('this callback execution cell is single-use')
if CHECKOUT.exists() or LOCAL_ROOT.exists() or TERMINAL_ZIP.exists():
    raise FileExistsError('fresh checkout, local root, and terminal ZIP path required')
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with LOCAL_STDOUT.open('xb'):
    pass
with LOCAL_STDERR.open('xb'):
    pass

def run_logged(command, *, cwd=None, env=None, check=True):
    with LOCAL_STDOUT.open('ab') as stdout, LOCAL_STDERR.open('ab') as stderr:
        return subprocess.run(
            command, cwd=cwd, env=env, stdout=stdout, stderr=stderr, check=check,
        )

def git_value(*args):
    return subprocess.run(
        ['git', '-C', str(CHECKOUT), *args],
        check=True, capture_output=True, text=True,
    ).stdout.strip()

def write_notebook_failure(stage, error):
    if LOCAL_RESULT.exists():
        return
    payload = {
        'claim_ceiling': 'engineering_image_only_callback_n4_prepared_not_executed',
        'denominator': 4,
        'error': f'{type(error).__name__}: {error}',
        'producer_exact': PRODUCER_EXACT,
        'records': [],
        'science_denominator': 0,
        'stage': stage,
        'status': 'OPERATIONAL_BLOCKED',
    }
    with LOCAL_RESULT.open('xb') as sink:
        sink.write(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('ascii'))

def write_terminal_zip():
    global BLIND_CALLBACK_TERMINAL_ZIP_WRITES
    if BLIND_CALLBACK_TERMINAL_ZIP_WRITES != 0:
        raise RuntimeError('terminal ZIP publication may be attempted only once')
    BLIND_CALLBACK_TERMINAL_ZIP_WRITES += 1
    members = []
    if checkout_verified and PUBLIC_N4_CONFIG.is_file():
        members.append((PUBLIC_N4_CONFIG, PUBLIC_N4_CONFIG.name))
    members.extend([
        (LOCAL_RESULT, LOCAL_RESULT.name),
        (LOCAL_STDOUT, LOCAL_STDOUT.name),
        (LOCAL_STDERR, LOCAL_STDERR.name),
    ])
    current_images = sorted(CURRENT_RGB_DIR.glob('current_rgb_*.png')) if CURRENT_RGB_DIR.is_dir() else []
    if len(current_images) > 4:
        raise RuntimeError('runner emitted more than four current candidate RGBs')
    members.extend((path, path.name) for path in current_images)
    with zipfile.ZipFile(TERMINAL_ZIP, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
        for source, arcname in members:
            archive.write(source, arcname=arcname)

stage = 'environment_guard'
runner_env = None
completed = None
checkout_verified = False
notebook_error = None
root_key = ''
hf_token = ''
try:
    if not torch.cuda.is_available():
        raise RuntimeError('GPU required; fixed N=4 was not executed')
    if not CALIBRATION_RUNS.is_dir():
        raise FileNotFoundError('calibration-runs Drive input is absent')
    stage = 'detached_checkout'
    run_logged(['git', 'clone', REPO_URL, str(CHECKOUT)])
    run_logged(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT])
    if git_value('rev-parse', 'HEAD') != PRODUCER_EXACT:
        raise RuntimeError('detached producer exact differs')
    if git_value('branch', '--show-current') != '' or git_value('status', '--porcelain=v1') != '':
        raise RuntimeError('producer checkout must be detached and clean')
    stage = 'dependency_install'
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)])
    if (
        git_value('rev-parse', 'HEAD') != PRODUCER_EXACT
        or git_value('branch', '--show-current') != ''
        or git_value('status', '--porcelain=v1') != ''
    ):
        raise RuntimeError('producer exact, detached state, or clean state changed during installation')
    checkout_verified = True
    stage = 'import_validation'
    run_logged([
        sys.executable, '-c',
        'from experiments import run_blind_detection_v1 as r; r.load_callback_n4_config(r.REPO_ROOT)',
    ], cwd=CHECKOUT)
    stage = 'secret_validation'
    root_key = userdata.get('CEG_WM_ROOT_KEY')
    hf_token = userdata.get('HF_TOKEN')
    if not isinstance(root_key, str) or not root_key.strip():
        raise RuntimeError('CEG_WM_ROOT_KEY Colab Secret is required')
    if not isinstance(hf_token, str) or not hf_token.strip():
        raise RuntimeError('HF_TOKEN Colab Secret is required')
    secret_markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
    runner_env = {
        name: value for name, value in os.environ.items()
        if not any(marker in name.upper() for marker in secret_markers)
    }
    runner_env['CEG_WM_ROOT_KEY'] = root_key
    runner_env['HF_TOKEN'] = hf_token
    root_key = ''
    hf_token = ''
    stage = 'formal_runner'
    command = [
        sys.executable, '-m', 'experiments.run_blind_detection_v1', 'callback-n4',
        '--producer-exact', PRODUCER_EXACT,
        '--calibration-runs-root', str(CALIBRATION_RUNS),
        '--runtime-root', str(RUNTIME_ROOT),
        '--current-rgb-output-dir', str(CURRENT_RGB_DIR),
        '--result-output', str(LOCAL_RESULT),
    ]
    BLIND_CALLBACK_RUNNER_CALLS += 1
    completed = run_logged(command, cwd=CHECKOUT, env=runner_env, check=False)
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if BLIND_CALLBACK_RUNNER_CALLS != 1 or not LOCAL_RESULT.is_file():
        raise RuntimeError('formal callback runner result is absent')
    stage = 'terminal_zip_publication'
except BaseException as error:
    notebook_error = error
finally:
    root_key = ''
    hf_token = ''
    if runner_env is not None:
        runner_env.pop('CEG_WM_ROOT_KEY', None)
        runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if notebook_error is not None:
        write_notebook_failure(stage, notebook_error)
    if not LOCAL_RESULT.is_file():
        write_notebook_failure(stage, RuntimeError('callback result is absent'))
    write_terminal_zip()

public_result = json.loads(LOCAL_RESULT.read_text(encoding='ascii'))
print('CEGWM_BLIND_CALLBACK_PUBLISHED ' + json.dumps({
    'denominator': public_result.get('denominator'),
    'runner_returncode': None if completed is None else completed.returncode,
    'status': public_result.get('status'),
    'terminal_zip': str(TERMINAL_ZIP),
}, sort_keys=True))


In [ ]:
with zipfile.ZipFile(TERMINAL_ZIP, mode='r') as archive:
    member_names = sorted(archive.namelist())
    public_result = json.loads(archive.read('callback_result.json').decode('ascii'))
    print('CEGWM_BLIND_CALLBACK_READBACK ' + json.dumps({
        'denominator': public_result.get('denominator'),
        'members': member_names,
        'status': public_result.get('status'),
    }, sort_keys=True))
